In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


fetching data

In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/1.2df_txt_norm.csv"

def load_text_data(path):
    df = pd.read_csv(path)

    print("📊 Raw dataframe shape:", df.shape)

    texts = df["text"].dropna().astype(str).tolist()

    print("📄 Total non-null texts:", len(texts))
    print("🧪 Sample text length:", len(texts[0]) if texts else 0)

    return texts

documents = load_text_data(CSV_PATH)

📊 Raw dataframe shape: (5797, 4)
📄 Total non-null texts: 5797
🧪 Sample text length: 298


#chunk function

In [ ]:
def split_text_into_chunks(text, chunk_size=500, overlap=50):
    text = str(text)

    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start + chunk_size]

        if len(chunk.strip()) > 50:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

creating chunks

In [ ]:
all_chunks = []

total_docs = len(documents)

for doc_index, text in enumerate(documents):
    chunks = split_text_into_chunks(text)

    print(f"\n📄 Document {doc_index+1}/{total_docs}")
    print(f"   Original length: {len(text)}")
    print(f"   Chunks created: {len(chunks)}")

    for chunk_index, chunk in enumerate(chunks):
        all_chunks.append({
            "doc_id": doc_index,
            "chunk_id": chunk_index,
            "text": chunk
        })

print("\n✅ TOTAL CHUNKS:", len(all_chunks))

Streaming output truncated to the last 5000 lines.
   Original length: 2719
   Chunks created: 6

📄 Document 4549/5797
   Original length: 11
   Chunks created: 0

📄 Document 4550/5797
   Original length: 158869
   Chunks created: 353

📄 Document 4551/5797
   Original length: 11
   Chunks created: 0

📄 Document 4552/5797
   Original length: 266573
   Chunks created: 593

📄 Document 4553/5797
   Original length: 11
   Chunks created: 0

📄 Document 4554/5797
   Original length: 535721
   Chunks created: 1191

📄 Document 4555/5797
   Original length: 11
   Chunks created: 0

📄 Document 4556/5797
   Original length: 5937
   Chunks created: 14

📄 Document 4557/5797
   Original length: 30
   Chunks created: 0

📄 Document 4558/5797
   Original length: 21109
   Chunks created: 47

📄 Document 4559/5797
   Original length: 9
   Chunks created: 0

📄 Document 4560/5797
   Original length: 10840
   Chunks created: 24

📄 Document 4561/5797
   Original length: 9
   Chunks created: 0

📄 Document 4562/

In [ ]:
# all_chunks

loading embdding model

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("⚙️ Loading model on:", device)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

print("✅ Model loaded")

In [ ]:
texts = [item["text"] for item in all_chunks]

print("\n📦 Total texts for embedding:", len(texts))
# print("🧪 Sample chunk length:", len(texts[0]) if texts else 0)
# print("🧪 Sample chunk preview:", texts[0][:100])


📦 Total texts for embedding: 741595


apply embedding here each batch have 64 text and each text have 384 dimension

In [ ]:
BATCH_SIZE = 64

embeddings = []

total_batches = (len(texts) // BATCH_SIZE) + 1

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]

    print(f"\n Processing batch {i//BATCH_SIZE + 1}/{total_batches}")
    print(f"   Batch size: {len(batch)}")

    batch_embeddings = embedding_model.encode(
        batch,
        batch_size=BATCH_SIZE,
        show_progress_bar=False
    )

    print(f"   Embedding shape: {len(batch_embeddings)}")

    embeddings.extend(batch_embeddings)

print("\n✅ TOTAL EMBEDDINGS:", len(embeddings))


 Processing batch 1/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 2/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 3/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 4/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 5/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 6/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 7/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 8/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 9/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 10/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 11/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 12/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 13/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 14/11588
   Batch size: 64
   Embedding shape: 64

 Processing batch 15/11588
   Batch size: 

KeyboardInterrupt: 

connect meaning (text) with math (vectors).

In [ ]:
final_data = []

for item, vector in zip(all_chunks, embeddings):
    final_data.append({
        "id": f"{item['doc_id']}_{item['chunk_id']}",
        "text_length": len(item["text"]),
        "embedding_dim": len(vector),
        "text_preview": item["text"][:80],
        "embedding": vector
    })

print("\n📊 FINAL DATASET SIZE:", len(final_data))
print("🧪 Sample record:")
print(final_data[0])